In [1]:
from dataclasses import dataclass
from datetime import datetime
from enum import Enum
from typing import Dict, List, Optional
import random

# =========================================================
# RICH TERMINAL UI
# =========================================================

from rich.console import Console
from rich.table import Table
from rich.panel import Panel
from rich.align import Align
from rich.rule import Rule
from rich.text import Text
from rich.columns import Columns
from rich import box

console = Console()

# =========================================================
# UI HELPERS
# =========================================================

class UI:

    @staticmethod
    def titulo(texto):

        console.print()

        console.print(
            Panel.fit(
                Align.center(
                    f"[bold cyan]{texto}[/bold cyan]"
                ),
                border_style="cyan",
                padding=(1, 5)
            )
        )

    @staticmethod
    def secao(texto):

        console.print(
            Rule(
                f"[bold yellow]{texto}[/bold yellow]",
                style="yellow"
            )
        )

    @staticmethod
    def sucesso(texto):

        console.print(
            f"[bold green]✔ {texto}[/bold green]"
        )

    @staticmethod
    def alerta(texto):

        console.print(
            f"[bold red]⚠ {texto}[/bold red]"
        )

    @staticmethod
    def info(texto):

        console.print(
            f"[bold blue]ℹ {texto}[/bold blue]"
        )


# =========================================================
# ENUMS
# =========================================================

class StatusSensor(Enum):
    ONLINE = "ONLINE"
    OFFLINE = "OFFLINE"
    ALERTA = "ALERTA"
    FALHA = "FALHA"


class TipoSensor(Enum):
    TEMPERATURA_INTERNA = "temperatura_interna"
    TEMPERATURA_EXTERNA = "temperatura_externa"
    VELOCIDADE_VENTO = "velocidade_vento"
    GERACAO_SOLAR = "geracao_solar"
    GERACAO_EOLICA = "geracao_eolica"
    CONSUMO_ENERGETICO = "consumo_energetico"
    ESTADO_MODULOS = "estado_modulos"
    INTEGRIDADE_ESTRUTURAL = "integridade_estrutural"


# =========================================================
# MODELO
# =========================================================

@dataclass
class LeituraSensor:
    sensor_id: str
    tipo: TipoSensor
    valor: float
    timestamp: datetime


# =========================================================
# CLASSE SENSOR
# =========================================================

class Sensor:

    def __init__(
        self,
        sensor_id: str,
        tipo: TipoSensor,
        limite_min: float,
        limite_max: float
    ):

        self.sensor_id = sensor_id
        self.tipo = tipo
        self.limite_min = limite_min
        self.limite_max = limite_max

        self.status = StatusSensor.ONLINE

        self.ultima_leitura: Optional[
            LeituraSensor
        ] = None

    # =====================================================
    # LEITURA
    # =====================================================

    def gerar_leitura(self) -> float:

        faixa = {

            TipoSensor.TEMPERATURA_INTERNA:
                (18, 32),

            TipoSensor.TEMPERATURA_EXTERNA:
                (-80, 40),

            TipoSensor.VELOCIDADE_VENTO:
                (0, 180),

            TipoSensor.GERACAO_SOLAR:
                (0, 1000),

            TipoSensor.GERACAO_EOLICA:
                (0, 700),

            TipoSensor.CONSUMO_ENERGETICO:
                (50, 1200),

            TipoSensor.ESTADO_MODULOS:
                (0, 100),

            TipoSensor.INTEGRIDADE_ESTRUTURAL:
                (0, 100)
        }

        minimo, maximo = faixa[self.tipo]

        return round(
            random.uniform(minimo, maximo),
            2
        )


# =========================================================
# SENSOR MANAGER
# =========================================================

class SensorManager:

    def __init__(self):

        self.sensores: Dict[
            str,
            Sensor
        ] = {}

        self.historico: List[
            LeituraSensor
        ] = []

        self.alertas: List[str] = []

        UI.titulo(
            "SENSOR MANAGER INICIALIZADO"
        )

    # =====================================================
    # REGISTRO
    # =====================================================

    def registrar_sensor(
        self,
        sensor_id: str,
        tipo: TipoSensor,
        limite_min: float,
        limite_max: float
    ):

        sensor = Sensor(
            sensor_id,
            tipo,
            limite_min,
            limite_max
        )

        self.sensores[sensor_id] = sensor

        tabela = Table(
            title="Novo Sensor Registrado",
            box=box.ROUNDED
        )

        tabela.add_column(
            "Sensor",
            style="cyan"
        )

        tabela.add_column(
            "Tipo",
            style="magenta"
        )

        tabela.add_column(
            "Faixa",
            style="green"
        )

        tabela.add_row(
            sensor_id,
            tipo.value,
            f"{limite_min} → {limite_max}"
        )

        console.print(tabela)

    # =====================================================
    # COLETA
    # =====================================================

    def coletar_dados(self):

        UI.titulo(
            "COLETA DE TELEMETRIA"
        )

        tabela = Table(
            title="Leituras dos Sensores",
            box=box.HEAVY_EDGE,
            show_lines=True
        )

        tabela.add_column(
            "Sensor",
            style="cyan"
        )

        tabela.add_column(
            "Tipo",
            style="magenta"
        )

        tabela.add_column(
            "Valor",
            justify="right"
        )

        tabela.add_column(
            "Timestamp",
            style="green"
        )

        for sensor in self.sensores.values():

            valor = sensor.gerar_leitura()

            leitura = LeituraSensor(
                sensor_id=sensor.sensor_id,
                tipo=sensor.tipo,
                valor=valor,
                timestamp=datetime.now()
            )

            sensor.ultima_leitura = leitura

            self.historico.append(leitura)

            tabela.add_row(
                sensor.sensor_id,
                sensor.tipo.value,
                f"{valor:.2f}",
                leitura.timestamp.strftime(
                    "%H:%M:%S"
                )
            )

        console.print(tabela)

        UI.sucesso(
            "Coleta concluída"
        )

    # =====================================================
    # VALIDAÇÃO
    # =====================================================

    def validar_dados(self):

        UI.titulo(
            "VALIDAÇÃO OPERACIONAL"
        )

        tabela = Table(
            title="Status dos Sensores",
            box=box.DOUBLE_EDGE,
            show_lines=True
        )

        tabela.add_column(
            "Sensor",
            style="cyan"
        )

        tabela.add_column(
            "Valor",
            justify="right"
        )

        tabela.add_column(
            "Faixa",
            style="yellow"
        )

        tabela.add_column(
            "Status",
            justify="center"
        )

        for sensor in self.sensores.values():

            leitura = sensor.ultima_leitura

            if leitura is None:
                continue

            valor = leitura.valor

            faixa = (
                f"{sensor.limite_min}"
                f" → "
                f"{sensor.limite_max}"
            )

            # =========================================
            # SENSOR OK
            # =========================================

            if sensor.limite_min <= valor <= sensor.limite_max:

                sensor.status = StatusSensor.ONLINE

                status = (
                    "[green]ONLINE[/green]"
                )

            # =========================================
            # ALERTA
            # =========================================

            else:

                sensor.status = StatusSensor.ALERTA

                status = (
                    "[red]ALERTA[/red]"
                )

                alerta = (
                    f"{sensor.sensor_id} "
                    f"fora da faixa operacional "
                    f"({valor:.2f})"
                )

                self.alertas.append(alerta)

            tabela.add_row(
                sensor.sensor_id,
                f"{valor:.2f}",
                faixa,
                status
            )

        console.print(tabela)

    # =====================================================
    # MONITORAMENTO
    # =====================================================

    def monitorar_integridade(self):

        UI.titulo(
            "MONITORAMENTO DA INTEGRIDADE"
        )

        paineis = []

        for sensor in self.sensores.values():

            leitura = sensor.ultima_leitura

            if leitura is None:

                sensor.status = StatusSensor.OFFLINE

            cor = "green"

            if sensor.status == StatusSensor.ALERTA:
                cor = "yellow"

            if sensor.status == StatusSensor.OFFLINE:
                cor = "red"

            painel = Panel(

                (
                    f"[bold cyan]{sensor.sensor_id}[/bold cyan]\n\n"

                    f"[white]Tipo:[/white] "
                    f"{sensor.tipo.value}\n"

                    f"[white]Status:[/white] "
                    f"[{cor}]"
                    f"{sensor.status.value}"
                    f"[/{cor}]\n"

                    f"[white]Último Valor:[/white] "
                    f"{leitura.valor if leitura else 'N/A'}"
                ),

                title="Sensor",
                border_style=cor,
                width=35

            )

            paineis.append(painel)

        console.print(
            Columns(paineis)
        )

    # =====================================================
    # ALERTAS
    # =====================================================

    def exibir_alertas(self):

        UI.titulo(
            "CENTRAL DE ALERTAS"
        )

        if not self.alertas:

            painel = Panel.fit(
                "[bold green]"
                "Nenhum alerta ativo"
                "[/bold green]",
                border_style="green"
            )

            console.print(painel)

            return

        tabela = Table(
            title="Alertas Operacionais",
            box=box.HEAVY,
            show_lines=True
        )

        tabela.add_column(
            "ID",
            justify="center"
        )

        tabela.add_column(
            "Descrição",
            style="red"
        )

        for i, alerta in enumerate(
            self.alertas,
            start=1
        ):

            tabela.add_row(
                str(i),
                alerta
            )

        console.print(tabela)


# =========================================================
# INICIALIZAÇÃO
# =========================================================

UI.titulo(
    "SISTEMA AURORA - CAMADA SENSORIAL"
)

sensor_manager = SensorManager()

# =========================================================
# REGISTROS
# =========================================================

sensor_manager.registrar_sensor(
    "TEMP-INT-01",
    TipoSensor.TEMPERATURA_INTERNA,
    18,
    30
)

sensor_manager.registrar_sensor(
    "TEMP-EXT-01",
    TipoSensor.TEMPERATURA_EXTERNA,
    -100,
    50
)

sensor_manager.registrar_sensor(
    "VENTO-01",
    TipoSensor.VELOCIDADE_VENTO,
    0,
    150
)

sensor_manager.registrar_sensor(
    "SOLAR-01",
    TipoSensor.GERACAO_SOLAR,
    0,
    1000
)

sensor_manager.registrar_sensor(
    "EOLICA-01",
    TipoSensor.GERACAO_EOLICA,
    0,
    700
)

sensor_manager.registrar_sensor(
    "ENERGIA-01",
    TipoSensor.CONSUMO_ENERGETICO,
    100,
    1000
)

sensor_manager.registrar_sensor(
    "MODULO-01",
    TipoSensor.ESTADO_MODULOS,
    70,
    100
)

sensor_manager.registrar_sensor(
    "ESTRUTURA-01",
    TipoSensor.INTEGRIDADE_ESTRUTURAL,
    80,
    100
)

# ========================================================= 
# EXECUÇÃO
# =========================================================

sensor_manager.coletar_dados()

sensor_manager.validar_dados()

sensor_manager.monitorar_integridade()

sensor_manager.exibir_alertas()

UI.titulo(
    "MONITORAMENTO FINALIZADO"
)

╭───────────────────────────────────────────╮
│                                           │
│     SISTEMA AURORA - CAMADA SENSORIAL     │
│                                           │
╰───────────────────────────────────────────╯

╭─────────────────────────────────────╮
│                                     │
│     SENSOR MANAGER INICIALIZADO     │
│                                     │
╰─────────────────────────────────────╯

            Novo Sensor Registrado             
╭─────────────┬─────────────────────┬─────────╮
│ Sensor      │ Tipo                │ Faixa   │
├─────────────┼─────────────────────┼─────────┤
│ TEMP-INT-01 │ temperatura_interna │ 18 → 30 │
╰─────────────┴─────────────────────┴─────────╯

             Novo Sensor Registrado              
╭─────────────┬─────────────────────┬───────────╮
│ Sensor      │ Tipo                │ Faixa     │
├─────────────┼─────────────────────┼───────────┤
│ TEMP-EXT-01 │ temperatura_externa │ -100 → 50 │
╰─────────────┴─────────────────────┴───────────╯

         Novo Sensor Registrado          
╭──────────┬──────────────────┬─────────╮
│ Sensor   │ Tipo             │ Faixa   │
├──────────┼──────────────────┼─────────┤
│ VENTO-01 │ velocidade_vento │ 0 → 150 │
╰──────────┴──────────────────┴─────────╯

...

╭──────────────────────────────────────╮
│                                      │
│     MONITORAMENTO DA INTEGRIDADE     │
│                                      │
╰──────────────────────────────────────╯

╭──────────── Sensor ─────────────╮ ╭──────────── Sensor ─────────────╮ ╭──────────── Sensor ─────────────╮
│ TEMP-INT-01                     │ │ TEMP-EXT-01                     │ │ VENTO-01                        │
│                                 │ │                                 │ │                                 │
│ Tipo: temperatura_interna       │ │ Tipo: temperatura_externa       │ │ Tipo: velocidade_vento          │
│ Status: ONLINE                  │ │ Status: ONLINE                  │ │ Status: ALERTA                  │
│ Último Valor: 21.62             │ │ Último Valor: 35.6              │ │ Último Valor: 150.8             │
╰─────────────────────────────────╯ ╰─────────────────────────────────╯ ╰─────────────────────────────────╯
╭──────────── Sensor ─────────────╮ ╭──────────── Sensor ─────────────╮ ╭──────────── Sensor ─────────────╮
│ SOLAR-01                        │ │ EOLICA-01                       │ │ ENERGIA-01                      │
│                                 │ │                                 │ │                                 │
│ Tipo: geracao_solar             │ │ Tipo: geracao_eolica            │ │ Tipo: consumo_energetico        │
│ Status: ONLINE                  │ │ Status: ONLINE                  │ │ Status: ONLINE                  │
│ Último Valor: 717.32            │ │ Último Valor: 334.22            │ │ Último Valor: 547.94            │
╰─────────────────────────────────╯ ╰─────────────────────────────────╯ ╰─────────────────────────────────╯
╭──────────── Sensor ─────────────╮ ╭──────────── Sensor ─────────────╮                                    
│ MODULO-01                       │ │ ESTRUTURA-01                    │                                    
│                                 │ │                                 │                                    
│ Tipo: estado_modulos            │ │ Tipo: integridade_estrutural    │                                    
│ Status: ONLINE                  │ │ Status: ONLINE                  │                                    
│ Último Valor: 91.33             │ │ Último Valor: 82.82             │                                    
╰─────────────────────────────────╯ ╰─────────────────────────────────╯

╭────────────────────────────╮
│                            │
│     CENTRAL DE ALERTAS     │
│                            │
╰────────────────────────────╯

                Alertas Operacionais                
┏━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ ID ┃ Descrição                                   ┃
┣━━━━╋━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┫
┃ 1  ┃ VENTO-01 fora da faixa operacional (150.80) ┃
┗━━━━┻━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛

╭──────────────────────────────────╮
│                                  │
│     MONITORAMENTO FINALIZADO     │
│                                  │
╰──────────────────────────────────╯

In [2]:
from dataclasses import dataclass
from datetime import datetime
from typing import Dict, List, Optional
import hashlib
import numpy as np

# =========================================================
# RICH TERMINAL UI
# =========================================================

from rich.console import Console
from rich.table import Table
from rich.panel import Panel
from rich.tree import Tree
from rich.rule import Rule
from rich import box

console = Console()

# =========================================================
# UI HELPERS
# =========================================================

class UI:

    @staticmethod
    def titulo(texto):

        console.print()

        console.print(
            Panel.fit(
                f"[bold cyan]{texto}[/bold cyan]",
                border_style="bright_blue",
                padding=(1, 5)
            )
        )

    @staticmethod
    def secao(texto):

        console.print(
            Rule(
                f"[bold yellow]{texto}[/bold yellow]",
                style="yellow"
            )
        )

    @staticmethod
    def sucesso(texto):

        console.print(
            f"[bold green]✔ {texto}[/bold green]"
        )

    @staticmethod
    def alerta(texto):

        console.print(
            f"[bold red]⚠ {texto}[/bold red]"
        )

    @staticmethod
    def info(texto):

        console.print(
            f"[bold blue]ℹ {texto}[/bold blue]"
        )


# =========================================================
# MODELOS
# =========================================================

@dataclass
class Telemetria:

    sensor_id: str

    tipo: str

    valor: float

    timestamp: datetime


# =========================================================
# NÓ DA ÁRVORE
# =========================================================

class TreeNode:

    def __init__(self, nome: str):

        self.nome = nome

        self.filhos: List["TreeNode"] = []

    def adicionar_filho(
        self,
        node: "TreeNode"
    ):

        self.filhos.append(node)


# =========================================================
# STORAGE MANAGER
# =========================================================

class StorageManager:
    """
    Camada responsável pelo armazenamento
    e organização dos dados da colônia Aurora.
    """

    def __init__(self):

        # ==============================================
        # HISTÓRICO
        # ==============================================

        self.historico: List[Telemetria] = []

        # ==============================================
        # VETORES
        # ==============================================

        self.buffer_temperaturas = np.array([])

        # ==============================================
        # MATRIZ
        # ==============================================

        self.matriz_energia = np.zeros((5, 24))

        # ==============================================
        # HASH TABLE
        # ==============================================

        self.hash_telemetria: Dict[
            str,
            List[Telemetria]
        ] = {}

        # ==============================================
        # ÁRVORE
        # ==============================================

        self.arvore_colonia = TreeNode(
            "Aurora"
        )

        self._inicializar_arvore()

        UI.titulo(
            "STORAGE MANAGER ONLINE"
        )

    # =====================================================
    # ÁRVORE
    # =====================================================

    def _inicializar_arvore(self):

        energia = TreeNode("Energia")

        sensores = TreeNode("Sensores")

        infraestrutura = TreeNode(
            "Infraestrutura"
        )

        energia.adicionar_filho(
            TreeNode("Solar")
        )

        energia.adicionar_filho(
            TreeNode("Eólica")
        )

        energia.adicionar_filho(
            TreeNode("Consumo")
        )

        sensores.adicionar_filho(
            TreeNode("Temperatura")
        )

        sensores.adicionar_filho(
            TreeNode("Vento")
        )

        infraestrutura.adicionar_filho(
            TreeNode("Módulos")
        )

        infraestrutura.adicionar_filho(
            TreeNode("Integridade")
        )

        self.arvore_colonia.adicionar_filho(
            energia
        )

        self.arvore_colonia.adicionar_filho(
            sensores
        )

        self.arvore_colonia.adicionar_filho(
            infraestrutura
        )

    # =====================================================
    # ARMAZENAMENTO
    # =====================================================

    def armazenar_telemetria(
        self,
        sensor_id: str,
        tipo: str,
        valor: float
    ):

        registro = Telemetria(

            sensor_id=sensor_id,

            tipo=tipo,

            valor=valor,

            timestamp=datetime.now()
        )

        # ==============================================
        # LISTA
        # ==============================================

        self.historico.append(
            registro
        )

        # ==============================================
        # HASH TABLE
        # ==============================================

        if sensor_id not in self.hash_telemetria:

            self.hash_telemetria[
                sensor_id
            ] = []

        self.hash_telemetria[
            sensor_id
        ].append(registro)

        # ==============================================
        # BUFFER
        # ==============================================

        if "temperatura" in tipo.lower():

            self.buffer_temperaturas = np.append(

                self.buffer_temperaturas,

                valor
            )

        # ==============================================
        # OUTPUT VISUAL
        # ==============================================

        tabela = Table(
            title="Registro Armazenado",
            box=box.ROUNDED
        )

        tabela.add_column(
            "Sensor",
            style="cyan"
        )

        tabela.add_column(
            "Tipo",
            style="yellow"
        )

        tabela.add_column(
            "Valor",
            style="green"
        )

        tabela.add_column(
            "Horário",
            style="magenta"
        )

        tabela.add_row(
            sensor_id,
            tipo,
            f"{valor:.2f}",
            registro.timestamp.strftime(
                "%H:%M:%S"
            )
        )

        console.print(tabela)

    # =====================================================
    # CONSULTA
    # =====================================================

    def consultar_historico(
        self,
        sensor_id: Optional[str] = None
    ) -> List[Telemetria]:

        UI.titulo(
            "CONSULTA DE HISTÓRICO"
        )

        if sensor_id:

            UI.info(
                f"Consulta HASH: {sensor_id}"
            )

            registros = self.hash_telemetria.get(
                sensor_id,
                []
            )

        else:

            UI.info(
                "Consulta Global"
            )

            registros = self.historico

        if not registros:

            UI.alerta(
                "Nenhum registro encontrado"
            )

            return []

        tabela = Table(
            title="Histórico Telemetria",
            box=box.DOUBLE_EDGE,
            show_lines=True
        )

        tabela.add_column(
            "Sensor",
            style="cyan"
        )

        tabela.add_column(
            "Tipo",
            style="yellow"
        )

        tabela.add_column(
            "Valor",
            style="green"
        )

        tabela.add_column(
            "Timestamp",
            style="magenta"
        )

        for item in registros:

            tabela.add_row(
                item.sensor_id,
                item.tipo,
                f"{item.valor:.2f}",
                item.timestamp.strftime(
                    "%d/%m %H:%M:%S"
                )
            )

        console.print(tabela)

        return registros

    # =====================================================
    # HASH
    # =====================================================

    def atualizar_hash(self):

        UI.titulo(
            "VERIFICAÇÃO DE INTEGRIDADE HASH"
        )

        tabela = Table(
            title="SHA-256",
            box=box.HEAVY,
            show_lines=True
        )

        tabela.add_column(
            "Sensor",
            style="cyan"
        )

        tabela.add_column(
            "Hash",
            style="green"
        )

        for sensor_id, registros in (
            self.hash_telemetria.items()
        ):

            conteudo = "".join(

                f"{r.valor}{r.timestamp}"

                for r in registros
            )

            hash_integridade = hashlib.sha256(

                conteudo.encode()

            ).hexdigest()

            tabela.add_row(
                sensor_id,
                hash_integridade[:32] + "..."
            )

        console.print(tabela)

    # =====================================================
    # MATRIZ
    # =====================================================

    def atualizar_matriz_energia(
        self,
        modulo: int,
        hora: int,
        consumo: float
    ):

        self.matriz_energia[
            modulo
        ][hora] = consumo

        UI.sucesso(
            f"Matriz atualizada "
            f"(Módulo={modulo}, "
            f"Hora={hora}, "
            f"Consumo={consumo:.2f})"
        )

    def exibir_matriz_energia(self):

        UI.titulo(
            "MATRIZ ENERGÉTICA"
        )

        tabela = Table(
            title="Consumo Energético",
            box=box.SQUARE
        )

        tabela.add_column(
            "Módulo",
            style="cyan"
        )

        for hora in range(24):

            tabela.add_column(
                str(hora),
                justify="center"
            )

        for i, linha in enumerate(
            self.matriz_energia
        ):

            tabela.add_row(

                f"M-{i}",

                *[
                    (
                        f"[green]{v:.0f}[/green]"
                        if v > 0
                        else "-"
                    )

                    for v in linha
                ]
            )

        console.print(tabela)

    # =====================================================
    # TEMPERATURAS
    # =====================================================

    def calcular_media_temperaturas(self):

        UI.titulo(
            "ANÁLISE TÉRMICA"
        )

        if len(self.buffer_temperaturas) == 0:

            UI.alerta(
                "Sem temperaturas registradas"
            )

            return 0

        media = np.mean(
            self.buffer_temperaturas
        )

        painel = Panel.fit(

            f"[bold green]"
            f"{media:.2f} °C"
            f"[/bold green]",

            title="Temperatura Média",

            border_style="green",

            padding=(1, 5)
        )

        console.print(painel)

        return media

    # =====================================================
    # ÁRVORE
    # =====================================================

    def exibir_hierarquia(self):

        UI.titulo(
            "HIERARQUIA DA COLÔNIA"
        )

        arvore = Tree(
            "[bold cyan]Aurora[/bold cyan]"
        )

        energia = arvore.add(
            "[yellow]Energia[/yellow]"
        )

        energia.add(
            "[green]Solar[/green]"
        )

        energia.add(
            "[green]Eólica[/green]"
        )

        energia.add(
            "[green]Consumo[/green]"
        )

        sensores = arvore.add(
            "[yellow]Sensores[/yellow]"
        )

        sensores.add(
            "[green]Temperatura[/green]"
        )

        sensores.add(
            "[green]Vento[/green]"
        )

        infraestrutura = arvore.add(
            "[yellow]Infraestrutura[/yellow]"
        )

        infraestrutura.add(
            "[green]Módulos[/green]"
        )

        infraestrutura.add(
            "[green]Integridade[/green]"
        )

        console.print(arvore)


# =========================================================
# EXEMPLO DE UTILIZAÇÃO
# =========================================================

console.clear()

UI.titulo(
    "AURORA SIGER • STORAGE MANAGER"
)

storage = StorageManager()

# =========================================================
# ARMAZENAMENTO
# =========================================================

storage.armazenar_telemetria(
    "TEMP-INT-01",
    "temperatura_interna",
    24.5
)

storage.armazenar_telemetria(
    "VENTO-01",
    "velocidade_vento",
    83.2
)

storage.armazenar_telemetria(
    "SOLAR-01",
    "geracao_solar",
    720.8
)

# =========================================================
# CONSULTA
# =========================================================

storage.consultar_historico(
    "TEMP-INT-01"
)

# =========================================================
# HASH
# =========================================================

storage.atualizar_hash()

# =========================================================
# MATRIZ
# =========================================================

storage.atualizar_matriz_energia(
    modulo=0,
    hora=10,
    consumo=520
)

storage.exibir_matriz_energia()

# =========================================================
# MÉDIA
# =========================================================

storage.calcular_media_temperaturas()

# =========================================================
# HIERARQUIA
# =========================================================

storage.exibir_hierarquia()

# =========================================================
# FINALIZAÇÃO
# =========================================================

console.print()

console.print(
    Panel.fit(
        "[bold cyan]"
        "STORAGE MANAGER FINALIZADO"
        "[/bold cyan]",
        border_style="cyan",
        padding=(1, 5)
    )
)

╭────────────────────────────────────────╮
│                                        │
│     AURORA SIGER • STORAGE MANAGER     │
│                                        │
╰────────────────────────────────────────╯

╭────────────────────────────────╮
│                                │
│     STORAGE MANAGER ONLINE     │
│                                │
╰────────────────────────────────╯

                  Registro Armazenado                   
╭─────────────┬─────────────────────┬───────┬──────────╮
│ Sensor      │ Tipo                │ Valor │ Horário  │
├─────────────┼─────────────────────┼───────┼──────────┤
│ TEMP-INT-01 │ temperatura_interna │ 24.50 │ 13:09:51 │
╰─────────────┴─────────────────────┴───────┴──────────╯

               Registro Armazenado                
╭──────────┬──────────────────┬───────┬──────────╮
│ Sensor   │ Tipo             │ Valor │ Horário  │
├──────────┼──────────────────┼───────┼──────────┤
│ VENTO-01 │ velocidade_vento │ 83.20 │ 13:09:51 │
╰──────────┴──────────────────┴───────┴──────────╯

              Registro Armazenado               
╭──────────┬───────────────┬────────┬──────────╮
│ Sensor   │ Tipo          │ Valor  │ Horário  │
├──────────┼───────────────┼────────┼──────────┤
│ SOLAR-01 │ geracao_solar │ 720.80 │ 13:09:51 │
╰──────────┴───────────────┴────────┴──────────╯

...

╭─────────────────────────╮
│                         │
│     ANÁLISE TÉRMICA     │
│                         │
╰─────────────────────────╯

╭─ Temperatura Média ─╮
│                     │
│     24.50 °C        │
│                     │
╰─────────────────────╯

╭───────────────────────────────╮
│                               │
│     HIERARQUIA DA COLÔNIA     │
│                               │
╰───────────────────────────────╯

Aurora
├── Energia
│   ├── Solar
│   ├── Eólica
│   └── Consumo
├── Sensores
│   ├── Temperatura
│   └── Vento
└── Infraestrutura
    ├── Módulos
    └── Integridade

╭────────────────────────────────────╮
│                                    │
│     STORAGE MANAGER FINALIZADO     │
│                                    │
╰────────────────────────────────────╯

In [ ]:
from dataclasses import dataclass
from datetime import datetime
from typing import Dict, List

import numpy as np

# =========================================================
# RICH TERMINAL UI
# =========================================================

from rich.console import Console
from rich.table import Table
from rich.panel import Panel
from rich.align import Align
from rich.columns import Columns
from rich.rule import Rule
from rich import box

console = Console()

# =========================================================
# MODELO
# =========================================================

@dataclass
class RegistroProcessado:
    """
    Estrutura responsável por representar um registro
    processado pelo Processing Engine.

    Atributos:
        sensor_id:
            Identificador único do sensor.

        tipo:
            Categoria operacional do sensor.

        valor:
            Valor numérico capturado.

        timestamp:
            Momento exato da ingestão.
    """

    sensor_id: str

    tipo: str

    valor: float

    timestamp: datetime


# =========================================================
# UI AUXILIAR
# =========================================================

class UI:
    """
    Camada utilitária responsável pela renderização
    visual no terminal utilizando Rich.

    Responsabilidades:
    - títulos;
    - seções;
    - mensagens operacionais;
    - painéis;
    - alertas;
    - feedback visual.
    """

    @staticmethod
    def titulo(texto):

        console.print()

        console.print(
            Panel.fit(
                Align.center(
                    f"[bold cyan]{texto}[/bold cyan]"
                ),
                border_style="cyan",
                padding=(1, 4)
            )
        )

    @staticmethod
    def secao(texto):

        console.print(
            Rule(
                f"[bold yellow]{texto}[/bold yellow]",
                style="yellow"
            )
        )

    @staticmethod
    def sucesso(texto):

        console.print(
            f"[bold green]✔ {texto}[/bold green]"
        )

    @staticmethod
    def erro(texto):

        console.print(
            f"[bold red]✖ {texto}[/bold red]"
        )

    @staticmethod
    def info(texto):

        console.print(
            f"[bold blue]ℹ {texto}[/bold blue]"
        )

    @staticmethod
    def warning(texto):

        console.print(
            f"[bold yellow]⚠ {texto}[/bold yellow]"
        )


# =========================================================
# PROCESSING ENGINE
# =========================================================

class ProcessingEngine:
    """
    Motor analítico responsável pelo processamento
    operacional da Colônia Aurora.

    =====================================================
    RESPONSABILIDADES
    =====================================================

    - ingestão de telemetria;
    - limpeza de dados;
    - validação operacional;
    - cálculos estatísticos;
    - agregações analíticas;
    - atualização matricial;
    - monitoramento de estados;
    - geração de métricas globais.

    =====================================================
    PIPELINE ANALÍTICO
    =====================================================

    Sensores
        ↓
    Validação
        ↓
    Limpeza
        ↓
    Vetorização NumPy
        ↓
    Estatísticas
        ↓
    Estados Operacionais

    =====================================================
    RECURSOS IMPLEMENTADOS
    =====================================================

    - média;
    - desvio padrão;
    - mínimo;
    - máximo;
    - mediana;
    - percentis;
    - agregações;
    - matriz operacional;
    - estados dinâmicos.

    =====================================================
    MELHORIAS PRIORITÁRIAS IMPLEMENTADAS
    =====================================================

    ✔ Correção de vetores vazios
    ✔ Percentis operacionais
    ✔ Mediana
    ✔ Variância
    ✔ Índice de estabilidade
    ✔ Proteção contra índices inválidos
    ✔ Correção de classificação crítica
    ✔ Estatísticas completas
    ✔ Diagnóstico energético
    ✔ Diagnóstico térmico
    """

    def __init__(self):

        # =================================================
        # HISTÓRICO PROCESSADO
        # =================================================

        self.dados_processados: List[
            RegistroProcessado
        ] = []

        # =================================================
        # VETORES NUMPY
        # =================================================

        self.vetor_temperaturas = np.array([])

        self.vetor_consumo = np.array([])

        self.vetor_solar = np.array([])

        self.vetor_eolico = np.array([])

        # =================================================
        # MATRIZ OPERACIONAL
        # =================================================

        self.matriz_operacional = np.zeros(
            (5, 24)
        )

        # =================================================
        # ESTATÍSTICAS
        # =================================================

        self.estatisticas: Dict[
            str,
            Dict
        ] = {}

        # =================================================
        # ESTADOS
        # =================================================

        self.estado_colonia = {

            "energia": "ESTÁVEL",

            "temperatura": "NORMAL",

            "estrutura": "SEGURA"
        }

        UI.titulo(
            "PROCESSING ENGINE INICIALIZADO"
        )

    # =====================================================
    # LIMPEZA
    # =====================================================

    def limpar_dados(
        self,
        registros: List[RegistroProcessado]
    ) -> List[RegistroProcessado]:
        """
        Remove registros inválidos do pipeline.

        Regras:
        - valores None são descartados;
        - NaN é descartado;
        - infinitos são descartados.

        Retorna:
            Lista contendo apenas registros válidos.
        """

        UI.secao(
            "LIMPEZA DE DADOS"
        )

        dados_limpos = []

        for registro in registros:

            if registro.valor is None:
                continue

            if np.isnan(registro.valor):
                continue

            if np.isinf(registro.valor):
                continue

            dados_limpos.append(
                registro
            )

        tabela = Table(
            title="Resultado da Limpeza",
            box=box.ROUNDED
        )

        tabela.add_column(
            "Recebidos",
            justify="center"
        )

        tabela.add_column(
            "Válidos",
            justify="center"
        )

        tabela.add_row(
            str(len(registros)),
            str(len(dados_limpos))
        )

        console.print(tabela)

        return dados_limpos

    # =====================================================
    # VALIDAÇÃO
    # =====================================================

    def validar_dados(
        self,
        registro: RegistroProcessado
    ) -> bool:
        """
        Realiza validação operacional.

        Critérios:
        - tipo conhecido;
        - valor dentro da faixa operacional.

        Retorna:
            True caso válido.
        """

        limites = {

            "temperatura_interna":
                (-20, 50),

            "temperatura_externa":
                (-150, 80),

            "velocidade_vento":
                (0, 250),

            "geracao_solar":
                (0, 1500),

            "geracao_eolica":
                (0, 1200),

            "consumo_energetico":
                (0, 5000),

            "integridade_estrutural":
                (0, 100)
        }

        if registro.tipo not in limites:

            UI.erro(
                f"Tipo desconhecido -> "
                f"{registro.tipo}"
            )

            return False

        minimo, maximo = limites[
            registro.tipo
        ]

        valido = (
            minimo <= registro.valor <= maximo
        )

        tabela = Table(
            title="Validação Operacional",
            box=box.SIMPLE_HEAVY
        )

        tabela.add_column("Sensor")

        tabela.add_column("Tipo")

        tabela.add_column("Valor")

        tabela.add_column("Status")

        status = (

            "[green]VÁLIDO[/green]"

            if valido

            else "[red]INVÁLIDO[/red]"
        )

        tabela.add_row(
            registro.sensor_id,
            registro.tipo,
            f"{registro.valor:.2f}",
            status
        )

        console.print(tabela)

        return valido

    # =====================================================
    # MÉDIA
    # =====================================================

    def calcular_media(
        self,
        vetor: np.ndarray
    ) -> float:
        """
        Calcula média aritmética.

        Retorna:
            Média do vetor.
        """

        UI.secao(
            "CÁLCULO DE MÉDIA"
        )

        if vetor.size == 0:

            UI.warning(
                "Vetor vazio"
            )

            return 0.0

        media = float(
            np.mean(vetor)
        )

        painel = Panel.fit(

            f"[bold cyan]{media:.2f}[/bold cyan]",

            title="Média",

            border_style="cyan"
        )

        console.print(painel)

        return media

    # =====================================================
    # MATRIZ
    # =====================================================

    def atualizar_matriz(
        self,
        modulo: int,
        horario: int,
        valor: float
    ):
        """
        Atualiza matriz operacional.

        Args:
            modulo:
                Índice do módulo.

            horario:
                Índice horário.

            valor:
                Valor operacional.
        """

        if not (0 <= modulo < 5):

            UI.erro(
                "Módulo inválido"
            )

            return

        if not (0 <= horario < 24):

            UI.erro(
                "Horário inválido"
            )

            return

        self.matriz_operacional[
            modulo
        ][horario] = valor

        tabela = Table(
            title="Atualização Matricial",
            box=box.MINIMAL_DOUBLE_HEAD
        )

        tabela.add_column("Módulo")

        tabela.add_column("Horário")

        tabela.add_column("Valor")

        tabela.add_row(
            str(modulo),
            str(horario),
            f"{valor:.2f}"
        )

        console.print(tabela)

    # =====================================================
    # ESTATÍSTICAS
    # =====================================================

    def processar_estatisticas(self):
        """
        Processa estatísticas globais.

        Métricas:
        - média;
        - mediana;
        - máximo;
        - mínimo;
        - desvio padrão;
        - variância;
        - percentil 90.
        """

        UI.titulo(
            "PROCESSAMENTO ESTATÍSTICO"
        )

        self.estatisticas = {}

        datasets = {

            "temperaturas":
                self.vetor_temperaturas,

            "consumo":
                self.vetor_consumo,

            "solar":
                self.vetor_solar,

            "eolico":
                self.vetor_eolico
        }

        for nome, vetor in datasets.items():

            if vetor.size == 0:
                continue

            self.estatisticas[nome] = {

                "media":
                    float(np.mean(vetor)),

                "mediana":
                    float(np.median(vetor)),

                "maximo":
                    float(np.max(vetor)),

                "minimo":
                    float(np.min(vetor)),

                "desvio_padrao":
                    float(np.std(vetor)),

                "variancia":
                    float(np.var(vetor)),

                "percentil_90":
                    float(
                        np.percentile(
                            vetor,
                            90
                        )
                    )
            }

        for categoria, valores in (
            self.estatisticas.items()
        ):

            tabela = Table(
                title=categoria.upper(),
                box=box.ROUNDED
            )

            tabela.add_column(
                "Métrica"
            )

            tabela.add_column(
                "Valor"
            )

            for chave, valor in (
                valores.items()
            ):

                tabela.add_row(
                    chave,
                    f"{valor:.2f}"
                )

            console.print(tabela)

    # =====================================================
    # AGREGAÇÕES
    # =====================================================

    def executar_agregacoes(
        self,
        registros: List[RegistroProcessado]
    ):
        """
        Executa agregações analíticas.

        Produz:
        - média;
        - soma;
        - quantidade;
        - máximo;
        - mínimo.
        """

        UI.titulo(
            "AGREGAÇÕES ANALÍTICAS"
        )

        agregacoes = {}

        for registro in registros:

            agregacoes.setdefault(
                registro.tipo,
                []
            ).append(
                registro.valor
            )

        tabela = Table(
            title="Resultado das Agregações",
            box=box.HEAVY
        )

        tabela.add_column("Tipo")

        tabela.add_column("Média")

        tabela.add_column("Soma")

        tabela.add_column("Máximo")

        tabela.add_column("Mínimo")

        tabela.add_column("Qtd")

        for tipo, valores in (
            agregacoes.items()
        ):

            tabela.add_row(

                tipo,

                f"{np.mean(valores):.2f}",

                f"{np.sum(valores):.2f}",

                f"{np.max(valores):.2f}",

                f"{np.min(valores):.2f}",

                str(len(valores))
            )

        console.print(tabela)

    # =====================================================
    # ESTADOS
    # =====================================================

    def atualizar_estados(self):
        """
        Atualiza estados operacionais.

        Estados:
        - NORMAL;
        - ALERTA;
        - CRÍTICA;
        - SOBRECARGA.
        """

        UI.titulo(
            "ATUALIZAÇÃO DOS ESTADOS"
        )

        media_temp = self.calcular_media(
            self.vetor_temperaturas
        )

        media_consumo = self.calcular_media(
            self.vetor_consumo
        )

        # =================================================
        # TEMPERATURA
        # =================================================

        if media_temp >= 35:

            self.estado_colonia[
                "temperatura"
            ] = "CRÍTICA"

        elif media_temp >= 28:

            self.estado_colonia[
                "temperatura"
            ] = "ALERTA"

        else:

            self.estado_colonia[
                "temperatura"
            ] = "NORMAL"

        # =================================================
        # ENERGIA
        # =================================================

        if media_consumo >= 3000:

            self.estado_colonia[
                "energia"
            ] = "SOBRECARGA"

        elif media_consumo >= 1800:

            self.estado_colonia[
                "energia"
            ] = "ALERTA"

        else:

            self.estado_colonia[
                "energia"
            ] = "ESTÁVEL"

        tabela = Table(
            title="Estado Atual da Colônia",
            box=box.DOUBLE
        )

        tabela.add_column(
            "Subsystem"
        )

        tabela.add_column(
            "Status"
        )

        for chave, valor in (
            self.estado_colonia.items()
        ):

            cor = "green"

            if valor in [
                "ALERTA",
                "SOBRECARGA"
            ]:
                cor = "yellow"

            if valor == "CRÍTICA":
                cor = "red"

            tabela.add_row(
                chave.upper(),
                f"[{cor}]{valor}[/{cor}]"
            )

        console.print(tabela)

    # =====================================================
    # ESTABILIDADE
    # =====================================================

    def calcular_estabilidade_energetica(self):
        """
        Calcula índice de estabilidade energética.

        Fórmula:

        Quanto menor o desvio padrão,
        maior a estabilidade.
        """

        UI.titulo(
            "ESTABILIDADE ENERGÉTICA"
        )

        if self.vetor_consumo.size == 0:

            UI.warning(
                "Sem dados energéticos"
            )

            return

        media = np.mean(
            self.vetor_consumo
        )

        desvio = np.std(
            self.vetor_consumo
        )

        indice = max(
            0,
            100 - (
                (desvio / media) * 100
            )
        )

        painel = Panel.fit(

            (
                f"[bold green]"
                f"{indice:.2f}%"
                f"[/bold green]\n\n"
                f"Desvio: {desvio:.2f}"
            ),

            title="Índice de Estabilidade",

            border_style="green"
        )

        console.print(painel)

    # =====================================================
    # INGESTÃO
    # =====================================================

    def adicionar_registro(
        self,
        sensor_id: str,
        tipo: str,
        valor: float
    ):
        """
        Realiza ingestão operacional.

        Fluxo:
        - validação;
        - armazenamento;
        - vetorização.
        """

        UI.secao(
            "INGESTÃO DE TELEMETRIA"
        )

        registro = RegistroProcessado(

            sensor_id=sensor_id,

            tipo=tipo,

            valor=valor,

            timestamp=datetime.now()
        )

        if not self.validar_dados(
            registro
        ):

            UI.erro(
                "Registro descartado"
            )

            return

        self.dados_processados.append(
            registro
        )

        # =================================================
        # VETORES ESPECIALIZADOS
        # =================================================

        if "temperatura" in tipo:

            self.vetor_temperaturas = np.append(
                self.vetor_temperaturas,
                valor
            )

        if "consumo" in tipo:

            self.vetor_consumo = np.append(
                self.vetor_consumo,
                valor
            )

        if "solar" in tipo:

            self.vetor_solar = np.append(
                self.vetor_solar,
                valor
            )

        if "eolica" in tipo:

            self.vetor_eolico = np.append(
                self.vetor_eolico,
                valor
            )

        painel = Panel.fit(

            (
                f"[bold green]"
                f"{sensor_id}"
                f"[/bold green]\n\n"
                f"Tipo: {tipo}\n"
                f"Valor: {valor:.2f}\n"
                f"Timestamp: "
                f"{registro.timestamp.strftime('%H:%M:%S')}"
            ),

            title="Registro Processado",

            border_style="green"
        )

        console.print(painel)


# =========================================================
# EXEMPLO DE UTILIZAÇÃO
# =========================================================

UI.titulo(
    "AURORA PROCESSING ENGINE"
)

engine = ProcessingEngine()

# =========================================================
# REGISTROS
# =========================================================

engine.adicionar_registro(
    "TEMP-INT-01",
    "temperatura_interna",
    24.8
)

engine.adicionar_registro(
    "TEMP-INT-02",
    "temperatura_interna",
    29.3
)

engine.adicionar_registro(
    "TEMP-INT-03",
    "temperatura_interna",
    36.1
)

engine.adicionar_registro(
    "ENERGIA-01",
    "consumo_energetico",
    2100
)

engine.adicionar_registro(
    "ENERGIA-02",
    "consumo_energetico",
    3200
)

engine.adicionar_registro(
    "SOLAR-01",
    "geracao_solar",
    840
)

engine.adicionar_registro(
    "SOLAR-02",
    "geracao_solar",
    920
)

engine.adicionar_registro(
    "EOLICA-01",
    "geracao_eolica",
    430
)

# =========================================================
# MÉDIA
# =========================================================

engine.calcular_media(
    engine.vetor_temperaturas
)

# =========================================================
# MATRIZ
# =========================================================

engine.atualizar_matriz(
    modulo=1,
    horario=14,
    valor=88.7
)

# =========================================================
# ESTATÍSTICAS
# =========================================================

engine.processar_estatisticas()

# =========================================================
# AGREGAÇÕES
# =========================================================

engine.executar_agregacoes(
    engine.dados_processados
)

# =========================================================
# ESTADOS
# =========================================================

engine.atualizar_estados()

# =========================================================
# ESTABILIDADE
# =========================================================

engine.calcular_estabilidade_energetica()

# =========================================================
# FINALIZAÇÃO
# =========================================================

UI.titulo(
    "PROCESSAMENTO FINALIZADO"
)